# GCP Secret Engine

In [ ]:
%%bash
# Authenticate to GCP (will open browser windows)
gcloud auth application-default login && gcloud auth login


In [ ]:
! open -a Podman\ Desktop

In [ ]:
import subprocess, os

# Auto-detect host IP address (DHCP-friendly)
for iface in ["en0", "en1", "en3", "en5"]:
    try:
        ip = subprocess.check_output(["ipconfig", "getifaddr", iface], text=True).strip()
        if ip:
            break
    except Exception:
        ip = None
if not ip:
    ip = "127.0.0.1"

os.environ["VAULT_IP"] = ip
os.environ["VAULT_ADDR"] = f"http://{ip}:8200"
os.environ["VAULT_TOKEN"] = "root"
os.environ["VAULT_PORT"] = "8200"
os.environ["VAULT_KMIP_PORT"] = "5696"

# Get current GCP project and active user account
project = "hc-0f488366ab0f42dba5be9a3d890"

# Primary source: gcloud config
user_account = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"],
    text=True,
    stderr=subprocess.DEVNULL,
).strip()

# Fallback: active authenticated account
if not user_account:
    try:
        user_account = subprocess.check_output(
            [
                "gcloud",
                "auth",
                "list",
                "--filter=status:ACTIVE",
                "--format=value(account)",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except Exception:
        user_account = ""

os.environ["GCP_PROJECT"] = project
if user_account:
    os.environ["GCP_USER_ACCOUNT"] = user_account

print(f"VAULT_IP:         {ip}")
print(f"VAULT_ADDR:       http://{ip}:8200")
print(f"GCP_PROJECT:      {project}")
print(f"GCP_USER_ACCOUNT: {user_account if user_account else '[not detected]'}")

### Create a Vault Server in Podman

In [ ]:
%%bash
# Change the path to your license file

export VAULT_LICENSE=$(cat vault.hclic)

# Refresh Vault docker image with latest version
podman pull hashicorp/vault-enterprise

podman run -d --rm --name vault-enterprise \
  --cap-add IPC_LOCK \
  -e "VAULT_DEV_ROOT_TOKEN_ID=${VAULT_TOKEN}" \
  -e "VAULT_DEV_LISTEN_ADDRESS=:${VAULT_PORT}" \
  -e "VAULT_LICENSE=${VAULT_LICENSE}" \
  -e "SKIP_SETCAP=true" \
  -p ${VAULT_IP}:${VAULT_KMIP_PORT}:${VAULT_KMIP_PORT} \
  -p ${VAULT_IP}:8200:${VAULT_PORT} \
  hashicorp/vault-enterprise:latest

### Check if Vault is running

In [ ]:
! podman ps

In [ ]:
%%bash
(podman ps --format 'table {{.Names}}\t{{.Status}}\t{{.Ports}}' && echo '---' && podman inspect vault-enterprise --format 'HostPortMapping: {{json .NetworkSettings.Ports}}' 2>/dev/null && podman inspect vault-enterprise --format 'ContainerIP: {{.NetworkSettings.IPAddress}}' 2>/dev/null && echo '---' && (ipconfig getifaddr en0 || ipconfig getifaddr en1))

## Enable the GCP secrets engine

In [ ]:
! vault secrets enable gcp

### Create `vault-admin` Service Account
The GCP secrets engine needs a service account with [specific permissions](https://developer.hashicorp.com/vault/docs/secrets/gcp#required-permissions) depending on the credential type:

**For rolesets** (project-level):
```
iam.serviceAccounts.create/delete/get/list/update
iam.serviceAccountKeys.create/delete/get/list
```

**For static accounts** (service-account-level):
```
iam.serviceAccounts.getAccessToken          # access_token secrets
iam.serviceAccountKeys.create/delete/get/list  # service_account_key secrets
```

**For rolesets/static accounts with bindings:**
```
resourcemanager.projects.getIamPolicy/setIamPolicy
```

We assign the following predefined roles to cover all scenarios:
- `roles/iam.serviceAccountAdmin` — manage service accounts (rolesets)
- `roles/iam.serviceAccountKeyAdmin` — manage service account keys (rolesets + rotate-root)
- `roles/resourcemanager.projectIamAdmin` — manage IAM policy bindings
- `roles/iam.serviceAccountTokenCreator` — generate access tokens (static accounts + impersonated accounts)

Additionally, **SA-level self-bindings** are needed for `rotate-root` (vault-admin must manage its own keys).

In [ ]:
%%bash
SA_NAME="vault-admin"
SA_EMAIL="${SA_NAME}@${GCP_PROJECT}.iam.gserviceaccount.com"

echo "Project:         $GCP_PROJECT"
echo "Service Account: $SA_EMAIL"

# Create service account (skip if already exists)
if gcloud iam service-accounts describe "$SA_EMAIL" &>/dev/null; then
    echo "Service account $SA_NAME already exists, skipping creation."
else
    gcloud iam service-accounts create "$SA_NAME" \
        --display-name="Vault Admin SA" \
        --description="Service account used by Vault GCP secrets engine"
    echo "Service account created."
fi

# Grant required project-level roles
for ROLE in roles/iam.serviceAccountAdmin roles/iam.serviceAccountKeyAdmin \
    roles/resourcemanager.projectIamAdmin roles/iam.serviceAccountTokenCreator; do
    echo "Granting $ROLE..."
    gcloud projects add-iam-policy-binding "$GCP_PROJECT" \
        --member="serviceAccount:${SA_EMAIL}" \
        --role="$ROLE" \
        --condition=None \
        --quiet 2>/dev/null
done

# Grant SA-level self-bindings so vault-admin can manage its own keys (needed for rotate-root)
echo ""
echo "Granting SA-level self-bindings on vault-admin..."
gcloud iam service-accounts add-iam-policy-binding "$SA_EMAIL" \
    --member="serviceAccount:${SA_EMAIL}" \
    --role="roles/iam.serviceAccountKeyAdmin" \
    --quiet 2>/dev/null
echo "Granted iam.serviceAccountKeyAdmin on self."

# Delete existing user-managed keys
echo ""
echo "Cleaning up old user-managed keys..."
for KEY_ID in $(gcloud iam service-accounts keys list --iam-account="$SA_EMAIL" \
    --managed-by=user --format="value(name)" 2>/dev/null); do
    gcloud iam service-accounts keys delete "$KEY_ID" \
        --iam-account="$SA_EMAIL" --quiet
    echo "  Deleted key: $KEY_ID"
done

# Create a new key
echo ""
echo "Creating new service account key..."
gcloud iam service-accounts keys create /tmp/vault-admin-key.json \
    --iam-account="$SA_EMAIL"
echo "Key saved to /tmp/vault-admin-key.json"

### Configure Vault GCP Secrets Engine
Pass the `vault-admin` service account key to the GCP secrets engine.

In [ ]:
%%bash
# Configure Vault GCP secrets engine with service account credentials
vault write gcp/config \
    credentials=@/tmp/vault-admin-key.json

echo "GCP secrets engine configured."

## Rotate Root Credentials

In [ ]:
! vault write -f gcp/config/rotate-root

## Static Accounts
A [static account](https://developer.hashicorp.com/vault/docs/secrets/gcp#static-accounts) maps a Vault path to an **existing** GCP service account.  
Vault can then generate **access tokens** or **service account keys** on demand for that service account.

We will create two static accounts:
1. **`app-token`** — generates OAuth2 **access tokens** (short-lived, auto-expire)
2. **`app-key`** — generates **service account keys** (JSON credentials for JWT signing)

### Create GCP Service Accounts for Static Accounts
We create two service accounts:
- `vault-static-token` — for the access token static account
- `vault-static-key` — for the service account key / JWT static account

In [ ]:
%%bash
ADMIN_SA="vault-admin@${GCP_PROJECT}.iam.gserviceaccount.com"

for SA_NAME in vault-static-token vault-static-key; do
    SA_EMAIL="${SA_NAME}@${GCP_PROJECT}.iam.gserviceaccount.com"
    echo "=== Service Account: $SA_NAME ==="

    # Create service account (skip if already exists)
    if gcloud iam service-accounts describe "$SA_EMAIL" &>/dev/null; then
        echo "Already exists, skipping creation."
    else
        gcloud iam service-accounts create "$SA_NAME" \
            --display-name="Vault Static SA ($SA_NAME)" \
            --description="Service account managed by Vault static account"
        echo "Created."
    fi

    # Grant Viewer role for testing (so we can verify credentials work)
    gcloud projects add-iam-policy-binding "$GCP_PROJECT" \
        --member="serviceAccount:${SA_EMAIL}" \
        --role="roles/viewer" \
        --condition=None \
        --quiet 2>/dev/null
    echo "Granted roles/viewer."

    # vault-admin needs tokenCreator + keyAdmin on these SAs
    gcloud iam service-accounts add-iam-policy-binding "$SA_EMAIL" \
        --member="serviceAccount:${ADMIN_SA}" \
        --role="roles/iam.serviceAccountTokenCreator" \
        --quiet 2>/dev/null
    gcloud iam service-accounts add-iam-policy-binding "$SA_EMAIL" \
        --member="serviceAccount:${ADMIN_SA}" \
        --role="roles/iam.serviceAccountKeyAdmin" \
        --quiet 2>/dev/null
    echo "Granted vault-admin tokenCreator + keyAdmin on $SA_NAME"
    echo ""
done

### Static Account — Access Tokens (`app-token`)
Configure a static account that generates **OAuth2 access tokens** for `vault-static-token`.  
Tokens are short-lived (default 1 hour) and automatically expire — no need to revoke.

In [ ]:
%%bash
SA_EMAIL="vault-static-token@${GCP_PROJECT}.iam.gserviceaccount.com"

vault write gcp/static-account/app-token \
    service_account_email="$SA_EMAIL" \
    secret_type="access_token" \
    token_scopes="https://www.googleapis.com/auth/cloud-platform"

echo ""
echo "Static account created. Reading config:"
vault read gcp/static-account/app-token

### Test — Read Access Token

In [ ]:
! vault read gcp/static-account/app-token/token

### Verification — Use Access Token with GCP APIs
Read the access token from Vault and use it to call GCP APIs.

In [ ]:
%%bash
echo "=== Static Account Access Token ==="
TOKEN_DATA=$(vault read -format=json gcp/static-account/app-token/token)
ACCESS_TOKEN=$(echo "$TOKEN_DATA" | jq -r '.data.token')
EXPIRES=$(echo "$TOKEN_DATA" | jq -r '.data.token_ttl')

echo "Token (first 20 chars): ${ACCESS_TOKEN:0:20}..."
echo "TTL: ${EXPIRES}s"
echo ""

# Use the token to list GCP projects
echo "=== List Projects (using static account token) ==="
curl -s -H "Authorization: Bearer ${ACCESS_TOKEN}" \
    "https://cloudresourcemanager.googleapis.com/v1/projects" \
    | jq '.projects[]? | {projectId, name, lifecycleState}' 2>/dev/null \
    || echo "No projects or insufficient permissions."

echo ""

# Use the token to list compute regions
echo "=== List Compute Regions ==="
curl -s -H "Authorization: Bearer ${ACCESS_TOKEN}" \
    "https://compute.googleapis.com/compute/v1/projects/${GCP_PROJECT}/regions" \
    | jq '[.items[]?.name] | .[:5]' 2>/dev/null \
    || echo "Unable to list regions."

echo ""

# Verify the token identity
echo "=== Token Info ==="
curl -s "https://oauth2.googleapis.com/tokeninfo?access_token=${ACCESS_TOKEN}" \
    | jq '{email, scope, expires_in}' 2>/dev/null

### Static Account — Service Account Keys / JWT (`app-key`)
Configure a static account that generates **service account keys** (JSON) for `vault-static-key`.  
These keys can be used to:
- Authenticate as the service account from external applications
- **Sign JWTs** for service-to-service authentication
- Generate ID tokens for identity-aware applications

> **Note:** Service account keys are long-lived credentials. Vault tracks and can revoke them.

In [ ]:
%%bash
SA_EMAIL="vault-static-key@${GCP_PROJECT}.iam.gserviceaccount.com"

vault write gcp/static-account/app-key \
    service_account_email="$SA_EMAIL" \
    secret_type="service_account_key"

echo ""
echo "Static account created. Reading config:"
vault read gcp/static-account/app-key

### Test — Read Service Account Key

In [ ]:
! vault read gcp/static-account/app-key/key

### Verification — Use Service Account Key to Generate a JWT
Read the service account key from Vault, authenticate with it, and use it to call GCP APIs.

In [ ]:
%%bash
set -euo pipefail

echo "=== Static Account Service Account Key ==="

# Try multiple times because newly created service-account keys can take a few seconds
# to become valid for OAuth token exchange in GCP.
MAX_ATTEMPTS=3
ATTEMPT=1
ACTIVATED=0

while [ "$ATTEMPT" -le "$MAX_ATTEMPTS" ]; do
  echo "Attempt ${ATTEMPT}/${MAX_ATTEMPTS}..."

  KEY_DATA=$(vault read -format=json gcp/static-account/app-key/key)

  # Decode key in jq to avoid platform-specific base64 flags and preserve exact JSON.
  echo "$KEY_DATA" | jq -r '.data.private_key_data | @base64d' > /tmp/vault-static-key.json

  echo "Key type:     $(jq -r '.type' /tmp/vault-static-key.json)"
  echo "Client email: $(jq -r '.client_email' /tmp/vault-static-key.json)"
  echo "Key ID:       $(jq -r '.private_key_id' /tmp/vault-static-key.json)"
  echo ""

  if gcloud auth activate-service-account --key-file=/tmp/vault-static-key.json --quiet; then
    ACTIVATED=1
    break
  fi

  echo "Activation failed; waiting for IAM propagation before retry..."
  ATTEMPT=$((ATTEMPT + 1))
  sleep 8
done

if [ "$ACTIVATED" -ne 1 ]; then
  echo "Failed to activate service account after ${MAX_ATTEMPTS} attempts."
  echo "You can inspect the last key file at /tmp/vault-static-key.json"
  exit 1
fi

echo "=== Authenticated Identity ==="
gcloud auth list --filter=status:ACTIVE --format="value(account)"
echo ""

echo "=== List Projects (using service account key) ==="
gcloud projects list --format="table(projectId, name, lifecycleState)" 2>/dev/null | head -10
echo ""

echo "=== Generate Access Token from Key (JWT flow) ==="
# gcloud uses the key to sign a JWT internally and exchanges it for an access token
ACCESS_TOKEN=$(gcloud auth print-access-token)
echo "Access token (first 20 chars): ${ACCESS_TOKEN:0:20}..."
echo ""

echo "=== Token Info ==="
curl -s "https://oauth2.googleapis.com/tokeninfo?access_token=${ACCESS_TOKEN}" \
    | jq '{email, scope, expires_in}' 2>/dev/null

# Clean up
SA_EMAIL=$(jq -r '.client_email' /tmp/vault-static-key.json)
rm -f /tmp/vault-static-key.json
gcloud auth revoke "$SA_EMAIL" --quiet 2>/dev/null || true

# Clean up

In [ ]:
%%bash
echo "=== Deleting Vault rolesets and static accounts ==="
vault delete gcp/static-account/app-token 2>/dev/null && echo "Deleted static account app-token" || true
vault delete gcp/static-account/app-key 2>/dev/null && echo "Deleted static account app-key" || true
vault secrets disable gcp 2>/dev/null && echo "Disabled mount gcp/" || true
echo "Vault resources cleaned up."

In [ ]:
%%bash
set -euo pipefail

echo "=== Deleting GCP Service Accounts ==="

# Ensure cleanup runs with your user account (not a revoked service-account login)
if [ -z "${GCP_USER_ACCOUNT:-}" ]; then
    GCP_USER_ACCOUNT=$(gcloud auth list --filter=status:ACTIVE --format="value(account)" 2>/dev/null | head -1 || true)
fi

# If still missing, re-login interactively (same flow as setup cell)
if [ -z "${GCP_USER_ACCOUNT:-}" ]; then
    echo "No active gcloud user detected. Launching browser login..."
    gcloud auth application-default login
    gcloud auth login
    GCP_USER_ACCOUNT=$(gcloud auth list --filter=status:ACTIVE --format="value(account)" 2>/dev/null | head -1 || true)
fi

if [ -n "${GCP_USER_ACCOUNT:-}" ]; then
    gcloud config set account "$GCP_USER_ACCOUNT" >/dev/null
else
    echo "ERROR: Could not detect a user account after login."
    exit 1
fi

gcloud config set project "$GCP_PROJECT" >/dev/null

echo "Active account: $(gcloud config get-value account 2>/dev/null)"
echo "Project:        $GCP_PROJECT"

for SA_NAME in vault-admin vault-static-token vault-static-key; do
    SA_EMAIL="${SA_NAME}@${GCP_PROJECT}.iam.gserviceaccount.com"
    echo ""
    echo "--- Deleting: $SA_NAME ---"

    # Skip cleanly if SA does not exist
    if ! gcloud iam service-accounts describe "$SA_EMAIL" --project="$GCP_PROJECT" >/dev/null 2>&1; then
        echo "  Service account $SA_NAME not found or already deleted."
        continue
    fi

    # Delete all user-managed keys
    for KEY_ID in $(gcloud iam service-accounts keys list --iam-account="$SA_EMAIL" \
        --managed-by=user --format="value(name)" --project="$GCP_PROJECT" 2>/dev/null); do
        gcloud iam service-accounts keys delete "$KEY_ID" \
            --iam-account="$SA_EMAIL" --quiet --project="$GCP_PROJECT" >/dev/null 2>&1 || true
        echo "  Deleted key: $(echo "$KEY_ID" | rev | cut -d'/' -f1 | rev)"
    done

    # Remove project-level IAM bindings
    for ROLE in roles/iam.serviceAccountAdmin roles/iam.serviceAccountKeyAdmin \
        roles/resourcemanager.projectIamAdmin roles/iam.serviceAccountTokenCreator roles/viewer; do
        gcloud projects remove-iam-policy-binding "$GCP_PROJECT" \
            --member="serviceAccount:${SA_EMAIL}" \
            --role="$ROLE" \
            --quiet >/dev/null 2>&1 || true
    done

    # Delete the service account
    if gcloud iam service-accounts delete "$SA_EMAIL" --quiet --project="$GCP_PROJECT" >/dev/null 2>&1; then
        echo "  Service account $SA_NAME deleted."
    else
        echo "  Failed to delete $SA_NAME (check permissions / org policy)."
        gcloud iam service-accounts describe "$SA_EMAIL" --project="$GCP_PROJECT" >/dev/null 2>&1 \
            && echo "  Service account still exists." \
            || echo "  Service account appears deleted."
    fi
done

echo ""
echo "=== GCP cleanup complete ==="

In [ ]:
%%bash
echo "=== Stopping Vault container ==="
podman stop vault-enterprise 2>/dev/null && echo "Vault container stopped." || echo "Vault container not running."
echo "=== All demo resources cleaned up ==="